# Autoregressive and Moving Average Models

## Autoregressive (AR) Processes

### Definition
An autoregressive process of order $p$, denoted $AR(p)$, models the current value of a time series as a linear function of its own past values plus a stochastic disturbance.

### Mathematical Formulation
The $AR(p)$ model is:
$$
y_t = \phi_1 y_{t-1} + \phi_2 y_{t-2} + \dots + \phi_p y_{t-p} + \varepsilon_t
$$
where:
- $y_t$ is the observed value at time $t$
- $\phi_i$ are autoregressive coefficients
- $\varepsilon_t$ is a white noise error term with $E[\varepsilon_t] = 0$ and $\text{Var}(\varepsilon_t) = \sigma^2$

In lag polynomial notation:
$$
\phi(L)y_t = \varepsilon_t
$$
with $\phi(L) = 1 - \phi_1 L - \dots - \phi_p L^p$ and $L y_t = y_{t-1}$.

### Stationarity
$AR(p)$ is weakly stationary if all roots of $\phi(z) = 0$ lie outside the unit circle. Stationarity ensures constant mean, variance, and autocovariance depending only on lag.

## Moving Average (MA) Processes

### Definition
A moving average process of order $q$, denoted MA(q), represents the current observation as a linear combination of current and past white noise terms.

### Mathematical Formulation
The $MA(q)$ model is:
$$
y_t = \varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \dots + \theta_q \varepsilon_{t-q}
$$
where:
- $\theta_i$ are moving average coefficients
- $\varepsilon_t$ is white noise

In lag polynomial notation:
$$
y_t = \theta(L) \varepsilon_t
$$
with $\theta(L) = 1 + \theta_1 L + \dots + \theta_q L^q$.

### Invertibility
MA(q) is invertible if all roots of $\theta(z) = 0$ lie outside the unit circle. Invertibility allows expressing the MA process as an infinite-order AR process.

# ARMA and ARIMA Models

## ARMA Processes

### Definition
The $ARMA(p, q)$ model combines autoregressive and moving average components to capture both persistence in the series and serially correlated shocks.

### Mathematical Formulation
The $ARMA(p, q)$ model is:
$$
y_t = \phi_1 y_{t-1} + \dots + \phi_p y_{t-p} + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \dots + \theta_q \varepsilon_{t-q}
$$
or equivalently:
$$
\phi(L) y_t = \theta(L) \varepsilon_t
$$
with
$\phi(L) = 1 - \phi_1 L - \dots - \phi_p L^p$ and
$\theta(L) = 1 + \theta_1 L + \dots + \theta_q L^q$.

### Properties
- Requires stationarity of the AR component.
- Requires invertibility of the MA component.
- Captures short-term dynamics and shocks simultaneously.

## ARIMA Processes

### Definition
The $ARIMA(p, d, q)$ model extends ARMA by introducing differencing to handle nonstationary series. The "I" stands for integration.

### Mathematical Formulation
If \(\Delta\) is the first difference operator, $\Delta y_t = y_t - y_{t-1}$, then:
$$
\phi(L) \Delta^d y_t = \theta(L) \varepsilon_t
$$
where:
- $d$ is the order of differencing
- $p$ is the autoregressive order
- $q$ is the moving average order

For $d = 1$:
$$
\phi(L) (1 - L) y_t = \theta(L) \varepsilon_t
$$

### Interpretation
- $d = 0$: $ARMA(p, q)$
- $d = 1$: first-differenced process, model for differences
- $d > 1$: higher-order differencing to achieve stationarity

### Practical Use
ARIMA models are widely used for forecasting economic and financial time series that exhibit trends or unit roots. Model identification typically involves:
- examining autocorrelation and partial autocorrelation functions,
- selecting $p$, $d$, and $q$,
- estimating parameters,
- validating residuals for whiteness.

# Summary
- AR models use past values of the series.
- MA models use past shocks.
- ARMA combines both to model stationary series.
- ARIMA adds differencing to model nonstationary series that can be made stationary through integration.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA

# Dati reali: serie storica dei sunspots
data = sm.datasets.sunspots.load_pandas().data
data.index = pd.RangeIndex(start=1700, stop=1700 + len(data))
serie = data["SUNACTIVITY"].astype(float)

# Fit ARMA(2,2) come ARIMA con d=0
arma = ARIMA(serie, order=(2, 0, 2)).fit()

# Fit ARIMA(2,1,2)
arima = ARIMA(serie, order=(2, 1, 2)).fit()

print("ARMA(2,2) summary")
print(arma.summary())
print("\nARIMA(2,1,2) summary")
print(arima.summary())

# Grafico dei fitted values
fig, axes = plt.subplots(2, 1, figsize=(12, 9))
serie.plot(ax=axes[0], label="Osservato")
arma.fittedvalues.plot(ax=axes[0], label="ARMA fitted")
axes[0].set_title("ARMA(2,2) fit")
axes[0].legend()

serie.plot(ax=axes[1], label="Osservato")
arima.fittedvalues.plot(ax=axes[1], label="ARIMA fitted")
axes[1].set_title("ARIMA(2,1,2) fit")
axes[1].legend()

plt.tight_layout()
plt.show()

# Previsione
steps = 36
fc_arma = arma.get_forecast(steps=steps)
fc_arima = arima.get_forecast(steps=steps)

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
serie.plot(ax=axes[0], label="Osservato")
fc_arma.predicted_mean.plot(ax=axes[0], label="ARMA forecast")
ci_arma = fc_arma.conf_int()
axes[0].fill_between(fc_arma.predicted_mean.index,
                     ci_arma.iloc[:, 0],
                     ci_arma.iloc[:, 1],
                     color="lightblue", alpha=0.3)
axes[0].set_title("Forecast ARMA(2,2)")
axes[0].legend()

serie.plot(ax=axes[1], label="Osservato")
fc_arima.predicted_mean.plot(ax=axes[1], label="ARIMA forecast")
ci_arima = fc_arima.conf_int()
axes[1].fill_between(fc_arima.predicted_mean.index,
                     ci_arima.iloc[:, 0],
                     ci_arima.iloc[:, 1],
                     color="lightgreen", alpha=0.3)
axes[1].set_title("Forecast ARIMA(2,1,2)")
axes[1].legend()

plt.tight_layout()
plt.show()